# 03 - Bias and Fairness Analysis
**NovaCred Credit Application Governance Analysis**

## 1. Imports & Load Data 

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from scipy import stats
from dateutil import parser as date_parser


df = pd.read_csv("../data/cleaned_credit_applications.csv")
df_raw = df.copy()
df_raw.head()

,app_id,full_name,email,ssn,ip_address,gender,date_of_birth,zip_code,annual_income,credit_history_months,...,processing_timestamp,loan_purpose,notes,gender_original,date_of_birth_original,high_dti_flag,email_valid,completeness_score,completeness_pct,ssn_duplicate_flag
0,app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,23.0,...,2024-01-15T00:00:00Z,NaN,NaN,Male,2001-03-09,False,True,12,100.0,False
1,app_037,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,51.0,...,NaN,NaN,NaN,M,1992-03-31,False,True,12,100.0,False
2,app_215,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,61000.0,41.0,...,NaN,vacation,NaN,Male,1989-10-24,False,True,12,100.0,False
3,app_024,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077.0,103000.0,70.0,...,NaN,NaN,NaN,Male,1983-04-25,False,True,12,100.0,False
4,app_184,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080.0,57000.0,14.0,...,2024-01-15T00:00:00Z,NaN,NaN,M,1999-05-21,False,True,12,100.0,False


---
## 2. PII Field Identification & Classification

Under GDPR Article 4(1), personal data is any information that relates to an identified or identifiable natural person. The analysis below distinguishes between **direct identifiers** (unambiguous linkage to an individual), **quasi-identifiers** (combinatorial re-identification risk), and **sensitive attributes** as defined by Article 9.

> **Reference:** GDPR Art. 4(1); Recital 26 (anonymisation standard); WP29/EDPB Opinion 05/2014 on anonymisation techniques.

In [4]:
#  PII Classification 

PII_CATALOGUE = [
    # ── Direct Identifiers ────────────────────────────────────────────────────
    {
        'column'          : 'full_name',
        'pii_category'    : 'Direct Identifier',
        'gdpr_article'    : 'Art. 4(1)',
        'risk_level'      : 'HIGH',
        'treatment'       : 'Pseudonymise with HMAC-SHA256; remove from analytical layer',
        'retention'       : '5 years post-contract closure (AML obligation)',
    },
    {
        'column'          : 'email',
        'pii_category'    : 'Direct Identifier',
        'gdpr_article'    : 'Art. 4(1), Art. 5(1)(c) — data minimisation',
        'risk_level'      : 'HIGH',
        'treatment'       : 'Pseudonymise for analytics; plaintext only in CRM with access controls',
        'retention'       : 'Duration of consent or contractual relationship',
    },
    {
        'column'          : 'ssn',
        'pii_category'    : 'Direct Identifier — Government ID',
        'gdpr_article'    : 'Art. 87; Recital 75 — high-risk identifier',
        'risk_level'      : 'CRITICAL',
        'treatment'       : 'Tokenise; never expose in analytical datasets; access restricted to compliance team only',
        'retention'       : '5 years post-contract (AML/KYC Directive 2015/849)',
    },
    {
        'column'          : 'ip_address',
        'pii_category'    : 'Online Identifier',
        'gdpr_article'    : 'Art. 4(1); Recital 30; CJEU C-582/14 (Breyer ruling)',
        'risk_level'      : 'MEDIUM',
        'treatment'       : 'Mask last octet (x.x.x.0); retain masked version max 90 days for fraud detection',
        'retention'       : '90 days — legitimate interest (Art. 6(1)(f)); LIA required',
    },
    # ── Quasi-Identifiers ────────────────────────────────────────────────────
    {
        'column'          : 'date_of_birth',
        'pii_category'    : 'Direct / Quasi-Identifier',
        'gdpr_article'    : 'Art. 4(1), Art. 5(1)(c) — minimisation',
        'risk_level'      : 'HIGH',
        'treatment'       : 'Generalise to age band (e.g. 25–34) for modelling; retain full DOB only where legally required',
        'retention'       : 'Governed by contractual/legal necessity',
    },
    {
        'column'          : 'date_of_birth_original',
        'pii_category'    : 'Direct / Quasi-Identifier (raw source field)',
        'gdpr_article'    : 'Art. 4(1), Art. 5(1)(e) — storage limitation',
        'risk_level'      : 'HIGH',
        'treatment'       : 'Delete after transformation to date_of_birth; no justification for retaining both fields',
        'retention'       : 'Should not persist beyond initial ingestion pipeline',
    },
    {
        'column'          : 'zip_code',
        'pii_category'    : 'Quasi-Identifier',
        'gdpr_article'    : 'Recital 26 — re-identification risk when combined with other fields',
        'risk_level'      : 'MEDIUM',
        'treatment'       : 'Aggregate to regional level for analytics; validate k-anonymity (k ≥ 5)',
        'retention'       : 'Duration of analytical purpose',
    },
    {
        'column'          : 'gender',
        'pii_category'    : 'Quasi-Identifier / Art. 9 proximity',
        'gdpr_article'    : 'Art. 9(1) — may reveal gender identity; Art. 5(1)(b) — purpose limitation',
        'risk_level'      : 'HIGH',
        'treatment'       : 'Assess necessity for credit model; if retained, document explicit lawful basis and conduct DPIA',
        'retention'       : 'Requires explicit consent (Art. 9(2)(a)) or statutory exception',
    },
    {
        'column'          : 'processing_timestamp',
        'pii_category'    : 'Metadata / Indirect Identifier',
        'gdpr_article'    : 'Art. 4(1) — identifiable when linked to applicant record',
        'risk_level'      : 'LOW',
        'treatment'       : 'Retain for audit trail (Art. 5(2) accountability); truncate to date only in analytical layer',
        'retention'       : '7 years for audit purposes',
    },
]

pii_df = pd.DataFrame(PII_CATALOGUE)

print(f"PII fields identified: {len(pii_df)}")
print(f"\nRisk distribution:")
print(pii_df['risk_level'].value_counts().to_string())
print()
print(pii_df[['column', 'pii_category', 'risk_level', 'gdpr_article']].to_string(index=False))

PII fields identified: 9

Risk distribution:
risk_level
HIGH        5
MEDIUM      2
CRITICAL    1
LOW         1

                column                                 pii_category risk_level                                                              gdpr_article
             full_name                            Direct Identifier       HIGH                                                                 Art. 4(1)
                 email                            Direct Identifier       HIGH                               Art. 4(1), Art. 5(1)(c) — data minimisation
                   ssn            Direct Identifier — Government ID   CRITICAL                                Art. 87; Recital 75 — high-risk identifier
            ip_address                            Online Identifier     MEDIUM                      Art. 4(1); Recital 30; CJEU C-582/14 (Breyer ruling)
         date_of_birth                    Direct / Quasi-Identifier       HIGH                                    Art. 4(1

---
## 3. Privacy Techniques

GDPR Recital 26 and Article 25 (Data Protection by Design) require that personal data be processed in a manner that reduces re-identification risk. This section demonstrates three complementary techniques:

| Technique | How it works | Reversible? | GDPR Status |
|-----------|-------------|-------------|-------------|
| **Pseudonymisation — Salted Hashing** | Replace identifiers with a keyed hash; salt prevents brute-force reversal | Yes (with key) | Still personal data — Art. 4(5) |
| **k-Anonymity — Generalisation & Suppression** | Generalise quasi-identifiers so each record is indistinguishable from ≥ k−1 others | No | Not personal data (if k is sufficient) |
| **Differential Privacy — Laplace Mechanism** | Add calibrated noise to outputs so an individual's presence cannot be inferred | No (mathematical guarantee) | Not personal data |
#### 
> **Warning** Pseudonymisation ≠ Anonymisation. A pseudonymised record can still be re-linked with the mapping key — it remains personal data under GDPR.

### Technique 1: Pseudonymisation via Salted Hashing

SHA-256 hashing with a secret salt replaces direct identifiers with pseudonyms. The salt makes brute-force reversal infeasible even if an attacker knows all possible inputs. Under GDPR Art. 4(5), pseudonymised data remains personal data. The salt must be stored separately and never included in the dataset.

In [5]:
import hashlib, secrets

# Generate a secret salt — in production this lives in a secrets manager / HSM
SALT = secrets.token_hex(16)
print(f"Salt (store separately — never in the dataset): {SALT[:8]}...  [REDACTED]\n")

def salted_hash(value: str, salt: str) -> str:
    """SHA-256 with salt — prevents brute-force reversal"""
    return hashlib.sha256((salt + str(value)).encode('utf-8')).hexdigest()[:16]

df_pseudo = df_raw.copy()

PII_COLS = ['full_name', 'email', 'ssn', 'ip_address']
existing_pii = [c for c in PII_COLS if c in df_pseudo.columns]

for col in existing_pii:
    df_pseudo[f'{col}_pseudonym'] = df_pseudo[col].apply(lambda x: salted_hash(str(x), SALT))
    df_pseudo.drop(columns=[col], inplace=True)

print(f"Columns pseudonymised: {existing_pii}")
print(f"Replaced with       : {[c + '_pseudonym' for c in existing_pii]}\n")

# Show before vs after side by side
pseudonym_cols = [c + '_pseudonym' for c in existing_pii]
before = df_raw[existing_pii].head()
after  = df_pseudo[pseudonym_cols].head()

print("Before pseudonymisation:")
display(before)
print("After pseudonymisation:")
display(after)

Salt (store separately — never in the dataset): d67b7587...  [REDACTED]

Columns pseudonymised: ['full_name', 'email', 'ssn', 'ip_address']
Replaced with       : ['full_name_pseudonym', 'email_pseudonym', 'ssn_pseudonym', 'ip_address_pseudonym']

Before pseudonymisation:


,full_name,email,ssn,ip_address
0,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155
1,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112
2,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250
3,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67
4,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105


After pseudonymisation:


,full_name_pseudonym,email_pseudonym,ssn_pseudonym,ip_address_pseudonym
0,20b0285cdea7b0ec,0e542dff582d5cb3,a813a3ba7db7098e,7df467115c86c516
1,ed15dac3ab7d8930,a223ab82ce9c5d4d,f297d0e6d896ca3d,93cf7de260e4eba5
2,201aa2889dac5faa,286e028689f67861,88f73fca601bd8f9,4f442f6deb114ab2
3,5287ae0b9418fcc8,2993f02145a846b2,bddd11d3b2abff97,effa540bda0854d1
4,e6ce54312e482d76,1608cfd0887cbfb8,cbd4a8f9c0a04300,dbbf6fc21742940c


### Technique 2: k-Anonymity — Generalisation & Suppression

Each record must be indistinguishable from at least k−1 others on the quasi-identifier combination. ZIP code, date of birth, and gender are generalised to reduce precision (e.g. 10036 → 100**, 1983 → 1980s). Records in groups that still fall below k after generalisation are suppressed.

In [10]:
df_kanon = df_pseudo.copy()

# Step 1: Generalise quasi-identifiers
# ZIP code: keep first 3 digits (postal district), mask last 2
if 'zip_code' in df_kanon.columns:
    df_kanon['zip_generalised'] = (
        df_kanon['zip_code'].astype(str).str[:3] + '**'
    )

# Date of birth → decade band
dob_col = next((c for c in df_kanon.columns if 'birth' in c.lower() and 'original' not in c.lower()), None)
if dob_col:
    birth_year = pd.to_datetime(df_kanon[dob_col], errors='coerce').dt.year
    df_kanon['dob_generalised'] = (birth_year // 10 * 10).astype('Int64').astype(str) + 's'

# Gender: keep as-is for k=2+, suppress if only 1 record in combination
# (suppression handled below)

# Step 2: Check k-anonymity on the quasi-identifier triple
k = 3
qi_cols = [c for c in ['zip_generalised', 'dob_generalised', 'gender'] if c in df_kanon.columns]

if len(qi_cols) >= 2:
    groups     = df_kanon.groupby(qi_cols, observed=True).size().reset_index(name='group_size')
    violations = groups[groups['group_size'] < k]
    satisfying = groups[groups['group_size'] >= k]

    print(f"Quasi-identifiers used : {qi_cols}")
    print(f"Target k               : {k}")
    print(f"Total combinations     : {len(groups)}")
    print(f"Satisfy k ≥ {k}          : {len(satisfying)} groups")
    print(f"Violations (< k)       : {len(violations)} groups (unique QI combinations below threshold)")
    print(f"Records in violations  : {violations['group_size'].sum()} rows (these are the records suppressed)")

    # ── Step 3: Suppression ───────────────────────────────────────────────
    df_kanon = df_kanon.merge(groups, on=qi_cols, how='left')
    n_suppressed = (df_kanon['group_size'] < k).sum()
    df_kanon = df_kanon[df_kanon['group_size'] >= k].drop(columns=['group_size'])

    print(f"\nSuppression summary:")
    print(f"  {len(violations)} groups × avg {violations['group_size'].mean():.1f} records/group = {n_suppressed} records removed")
    print(f"  Records retained : {len(df_kanon)} / {len(df_raw)} ({len(df_kanon)/len(df_raw):.1%})")
    print(f"\nBefore generalisation:")
    display(df_raw[["zip_code", "date_of_birth", "gender"]].head())
    print("After generalisation & suppression:")
    display(df_kanon[qi_cols].head())
else:
    print("⚠ Required quasi-identifier columns not found.")
    violations = pd.DataFrame()

Quasi-identifiers used : ['zip_generalised', 'dob_generalised', 'gender']
Target k               : 3
Total combinations     : 34
Satisfy k ≥ 3          : 18 groups
Violations (< k)       : 16 groups (unique QI combinations below threshold)
Records in violations  : 23 rows (these are the records suppressed)

Suppression summary:
  16 groups × avg 1.4 records/group = 23 records removed
  Records retained : 475 / 500 (95.0%)

Before generalisation:


,zip_code,date_of_birth,gender
0,10036.0,2001-03-09,Male
1,10032.0,1992-03-31,Male
2,10075.0,1989-10-24,Male
3,10077.0,1983-04-25,Male
4,10080.0,1999-05-21,Male


After generalisation & suppression:


,zip_generalised,dob_generalised,gender
0,100**,2000s,Male
1,100**,1990s,Male
2,100**,1980s,Male
3,100**,1980s,Male
4,100**,1990s,Male


### Technique 3: Differential Privacy — Laplace Mechanism

Calibrated Laplace noise is added to aggregate query outputs so that an individual's presence in the dataset cannot be inferred. The epsilon (ε) parameter controls the privacy-utility trade-off: lower ε means more noise and stronger privacy guarantees. Applied here to mean income and approval rate — the only statistics NovaCred should publish externally.

In [13]:
def laplace_mechanism(true_value: float, sensitivity: float, epsilon: float) -> float:
    """Add Laplace noise scaled to sensitivity/epsilon (Slide 32).
    
    sensitivity: max change one individual can cause in the query result.
    epsilon: privacy budget — lower = more noise = more privacy.
    """
    scale = sensitivity / epsilon
    noise = np.random.laplace(loc=0, scale=scale)
    return true_value + noise

# Query 1: Mean annual income
income_col = next((c for c in df_raw.columns if 'income' in c.lower()), None)

if income_col:
    true_mean_income = df_raw[income_col].mean()
    income_sensitivity = (df_raw[income_col].max() - df_raw[income_col].min()) / len(df_raw)

    print("Query: Mean Annual Income")
    print(f"  True answer           : €{true_mean_income:,.0f}")
    print(f"  Sensitivity           : €{income_sensitivity:,.2f}")
    print()
    print(f"  {'ε':<6} {'Noise Added':>15} {'Released Answer':>18} {'Abs Error':>12}")
    print(f"  {'-'*55}")
    for eps in [0.1, 0.5, 1.0, 5.0, 10.0]:
        np.random.seed(42)
        noisy = laplace_mechanism(true_mean_income, income_sensitivity, eps)
        error = abs(noisy - true_mean_income)
        privacy_label = '← high privacy' if eps == 0.1 else ('← low privacy' if eps == 10.0 else '')
        print(f"  ε={eps:<5} {noisy - true_mean_income:>+14,.0f}  €{noisy:>14,.0f}  ±€{error:>8,.0f}  {privacy_label}")

# Query 2: Approval rate (count-based query)
outcome_col = next((c for c in df_raw.columns if c.lower() in
                    ['loan_approved', 'credit_approved', 'approved', 'decision']), None)

if outcome_col:
    true_approval_rate = df_raw[outcome_col].astype(float).mean()
    rate_sensitivity = 1 / len(df_raw)

    print(f"\nQuery: Overall Loan Approval Rate")
    print(f"  True answer : {true_approval_rate:.3f} ({true_approval_rate:.1%})")
    print()
    print(f"  {'ε':<6} {'Released Rate':>15} {'Abs Error':>12}")
    print(f"  {'-'*36}")
    for eps in [0.1, 1.0, 10.0]:
        np.random.seed(42)
        noisy_rate = np.clip(laplace_mechanism(true_approval_rate, rate_sensitivity, eps), 0, 1)
        print(f"  ε={eps:<5} {noisy_rate:>14.3f}  ±{abs(noisy_rate - true_approval_rate):>8.4f}")

print("\n→ NovaCred should use ε=1.0 for published aggregate statistics (moderate privacy-utility trade-off).")
print("→ Individual-level data must NEVER be released — only DP aggregates")
      

# Before / after summary
if income_col:
    print('\nBefore (true values):')
    display(df_raw[[income_col]].describe().loc[['mean','min','max']].round(2))
    print('After (ε=1.0 — values that would be published externally):')
    np.random.seed(42)
    published = {
        'mean_income'   : round(laplace_mechanism(df_raw[income_col].mean(), income_sensitivity, 1.0), 2),
    }
    if outcome_col:
        np.random.seed(42)
        published['approval_rate'] = round(np.clip(laplace_mechanism(df_raw[outcome_col].astype(float).mean(), 1/len(df_raw), 1.0), 0, 1), 4)
    display(pd.DataFrame(published, index=['DP released value (ε=1.0)']))

Query: Mean Annual Income
  True answer           : €82,559
  Sensitivity           : €342.00

  ε          Noise Added    Released Answer    Abs Error
  -------------------------------------------------------
  ε=0.1             -988  €        81,571  ±€     988  ← high privacy
  ε=0.5             -198  €        82,361  ±€     198  
  ε=1.0              -99  €        82,460  ±€      99  
  ε=5.0              -20  €        82,539  ±€      20  
  ε=10.0             -10  €        82,549  ±€      10  ← low privacy

Query: Overall Loan Approval Rate
  True answer : 0.584 (58.4%)

  ε        Released Rate    Abs Error
  ------------------------------------
  ε=0.1            0.578  ±  0.0058
  ε=1.0            0.583  ±  0.0006
  ε=10.0           0.584  ±  0.0001

→ NovaCred should use ε=1.0 for published aggregate statistics (moderate privacy-utility trade-off).
→ Individual-level data must NEVER be released — only DP aggregates

Before (true values):


,annual_income
mean,82558.87
min,0.00
max,171000.00


After (ε=1.0 — values that would be published externally):


,mean_income,approval_rate
DP released value (ε=1.0),82460.06,0.5834


### Result after applying all three privacy techniques

✓ Direct identifiers  → replaced with salted hash pseudonyms (Technique 1)

✓ Quasi-identifiers   → generalised; rare combinations suppressed (Technique 2)

✓ Aggregate outputs   → DP noise applied before any publication (Technique 3)

In [17]:
# Drop remaining raw PII columns not yet removed
PII_DROP = ['full_name', 'name', 'email', 'phone', 'ssn',
            'national_id', 'date_of_birth', 'dob', 'address',
            'ip_address', 'zip_code', 'date_of_birth_original']

df_analytical = df_kanon.drop(
    columns=[c for c in PII_DROP if c in df_kanon.columns],
    errors='ignore'
)

FINAL_COL_ORDER = [
    'app_id', 'full_name_pseudonym', 'email_pseudonym', 'ssn_pseudonym',
    'ip_address_pseudonym', 'gender', 'dob_generalised', 'zip_generalised',
    'annual_income', 'credit_history_months', 'debt_to_income',
    'savings_balance', 'spending_total', 'spending_categories',
    'processing_timestamp', 'loan_purpose', 'loan_amount', 'loan_approved',
    'employment_type', 'gender_original', 'age', 'age_group',
]

# Only keep columns that actually exist in df_analytical
ordered_cols = [c for c in FINAL_COL_ORDER if c in df_analytical.columns]
# Append any remaining columns not explicitly listed
remaining = [c for c in df_analytical.columns if c not in ordered_cols]
df_analytical = df_analytical[ordered_cols + remaining]

print("Analytical dataset summary:")
print(f"  Original rows        : {len(df_raw)}")
print(f"  After suppression    : {len(df_analytical)} ({len(df_analytical)/len(df_raw):.1%} retained)")
print(f"  Original columns     : {len(df_raw.columns)}")
print(f"  Analytical columns   : {len(df_analytical.columns)}")
print(f"\n  PII columns removed  : {[c for c in RAW_PII_DROP if c in df_kanon.columns]}")
print(f"  Pseudonym columns    : {[c for c in df_analytical.columns if 'pseudonym' in c]}")
print(f"  Generalised QI       : {[c for c in df_analytical.columns if 'generalised' in c]}")
print()
df_analytical.head(3)

Analytical dataset summary:
  Original rows        : 500
  After suppression    : 475 (95.0% retained)
  Original columns     : 29
  Analytical columns   : 28

  PII columns removed  : ['date_of_birth', 'zip_code', 'date_of_birth_original']
  Pseudonym columns    : ['full_name_pseudonym', 'email_pseudonym', 'ssn_pseudonym', 'ip_address_pseudonym']
  Generalised QI       : ['dob_generalised', 'zip_generalised']



,app_id,full_name_pseudonym,email_pseudonym,ssn_pseudonym,ip_address_pseudonym,gender,dob_generalised,zip_generalised,annual_income,credit_history_months,...,spending_category_list,interest_rate,approved_amount,rejection_reason,notes,high_dti_flag,email_valid,completeness_score,completeness_pct,ssn_duplicate_flag
0,app_200,20b0285cdea7b0ec,0e542dff582d5cb3,a813a3ba7db7098e,7df467115c86c516,Male,2000s,100**,73000.0,23.0,...,Shopping|Rent|Alcohol,NaN,NaN,algorithm_risk_score,NaN,False,True,12,100.0,False
1,app_037,ed15dac3ab7d8930,a223ab82ce9c5d4d,f297d0e6d896ca3d,93cf7de260e4eba5,Male,1990s,100**,78000.0,51.0,...,Rent|Dining|Healthcare,NaN,NaN,algorithm_risk_score,NaN,False,True,12,100.0,False
2,app_215,201aa2889dac5faa,286e028689f67861,88f73fca601bd8f9,4f442f6deb114ab2,Male,1980s,100**,61000.0,41.0,...,Rent,3.7,59000.0,NaN,NaN,False,True,12,100.0,False


---
## Section 4 — GDPR Article Mapping

The following section maps NovaCred's credit data processing activities to specific GDPR obligations. Each processing activity is assessed across four key dimensions: **lawful basis**, **data minimisation**, **storage limitation**, and **right to erasure**.

In [22]:
# ── GDPR Article Mapping ───────────────────────────────────────────────────────

GDPR_MAPPING = [
    {
        'gdpr_article'          : 'Article 5(1)(a) — Lawfulness, Fairness, Transparency',
        'obligation'            : 'All processing must have a documented lawful basis; applicants must be informed via a clear privacy notice.',
        'novacred_finding'      : 'Dataset contains full_name, email, SSN, IP address and date of birth without a documented lawful basis per field. Consent is one valid basis but not the only one — contractual necessity (Art. 6(1)(b)) may apply to core credit decisioning.',
        'compliance_gap'        : 'Missing consent records and absence of a documented RoPA (Record of Processing Activities)',
        'recommended_action'    : 'Implement consent management platform; document all processing activities in RoPA (Art. 30); conduct DPIA before deploying credit scoring model.',
        'priority'              : 'CRITICAL',
    },
    {
        'gdpr_article'          : 'Article 5(1)(b) — Purpose Limitation',
        'obligation'            : 'Data collected for credit decisioning must not be repurposed (e.g., for marketing) without a new lawful basis.',
        'novacred_finding'      : 'Dataset includes full_name, phone, and email — fields beyond what is strictly necessary for automated credit scoring.',
        'compliance_gap'        : 'No documented purpose specification per processing activity; risk of scope creep',
        'recommended_action'    : 'Define and document processing purposes for each field; enforce purpose binding via data classification tags in the data catalogue.',
        'priority'              : 'HIGH',
    },
    {
        'gdpr_article'          : 'Article 5(1)(c) — Data Minimisation',
        'obligation'            : 'Only data that is adequate, relevant, and limited to what is necessary for the processing purpose should be collected.',
        'novacred_finding'      : 'Full DOB, SSN, and IP address are collected. For credit scoring, age band and postcode district may be sufficient.',
        'compliance_gap'        : 'Overcollection of precision PII beyond model requirements; date_of_birth_original retained alongside date_of_birth with no justification.',
        'recommended_action'    : 'Apply minimisation at collection point: collect age band instead of DOB; region instead of full address. Retain full identifiers only where AML/KYC law requires.',
        'priority'              : 'HIGH',
    },
    {
        'gdpr_article'          : 'Article 5(1)(e) — Storage Limitation',
        'obligation'            : 'Personal data must not be kept in identifiable form longer than necessary for the stated purpose.',
        'novacred_finding'      : 'No retention policy is encoded in the dataset schema. date_of_birth_original persists alongside the cleaned date_of_birth with no justification for retaining both.',
        'compliance_gap'        : 'Absence of automated deletion schedule; no documented retention periods by data category.',
        'recommended_action'    : 'Implement tiered retention: (1) Direct PII — delete 5 years post-contract; (2) Analytical pseudonyms — 7 years for AML compliance; (3) Anonymised aggregates — indefinite.',
        'priority'              : 'HIGH',
    },
    {
        'gdpr_article'          : 'Article 6(1) — Lawful Basis for Processing',
        'obligation'            : 'Each processing activity must be grounded in one of six legal bases (consent, contract, legal obligation, vital interests, public task, legitimate interests).',
        'novacred_finding'      : 'Credit decisioning: Art. 6(1)(b) (contractual necessity). Marketing communications: requires Art. 6(1)(a) (consent). Fraud detection via IP address: Art. 6(1)(f) (legitimate interests) — LIA required.',
        'compliance_gap'        : 'No documented LIA for fraud/IP processing; no separate consent flows for marketing vs. credit decisioning.',
        'recommended_action'    : 'Document a Legitimate Interests Assessment (LIA) for fraud detection use cases. Separate consent granularly per purpose in onboarding flow.',
        'priority'              : 'CRITICAL',
    },
    {
        'gdpr_article'          : 'Article 17 — Right to Erasure (Right to be Forgotten)',
        'obligation'            : 'Data subjects may request deletion where consent is withdrawn, processing is unlawful, or data is no longer necessary. Exceptions apply for legal obligations.',
        'novacred_finding'      : 'No erasure workflow is implemented. SSN and full PII persist in the raw dataset without a deletion mechanism.',
        'compliance_gap'        : 'No data subject request (DSR) management system; no ability to selectively delete records while preserving anonymised data.',
        'recommended_action'    : 'Build DSR workflow with 30-day SLA (Art. 12(3)). Use crypto-shredding — destroy the salt to render all pseudonyms permanently irreversible without physical deletion of each record.',
        'priority'              : 'CRITICAL',
    },
    {
        'gdpr_article'          : 'Article 22 — Automated Decision-Making',
        'obligation'            : 'Data subjects have the right not to be subject to solely automated decisions producing significant legal effects unless one of three exceptions applies.',
        'novacred_finding'      : 'loan_approved is a binary outcome constituting a significant legal/financial effect. If the model operates without human review, Art. 22 applies.',
        'compliance_gap'        : 'No evidence of human-in-the-loop for automated rejections; no documented explanation mechanism for adverse decisions.',
        'recommended_action'    : 'Implement mandatory human review for all rejections. Provide applicants with the right to explanation (Art. 22(3)) — document model logic in a Model Card. Log every decision with an auditable rationale.',
        'priority'              : 'CRITICAL',
    },
    {
        'gdpr_article'          : 'Article 25 — Data Protection by Design and by Default',
        'obligation'            : 'Technical and organisational measures must embed data protection into processing systems from the design stage.',
        'novacred_finding'      : 'Raw dataset contains full PII with no access controls or field-level encryption in the prototype pipeline.',
        'compliance_gap'        : 'No Privacy by Design documentation; no field-level access control; no encryption at rest demonstrated.',
        'recommended_action'    : 'Adopt layered data architecture: raw (encrypted, access-restricted) → pseudonymised → analytical. Enforce column-level security in the data warehouse. Conduct DPIA before model deployment.',
        'priority'              : 'HIGH',
    },
]

gdpr_df = pd.DataFrame(GDPR_MAPPING)
print(f"GDPR obligations mapped: {len(gdpr_df)}")
print(f"\nPriority breakdown:")
print(gdpr_df['priority'].value_counts().to_string())
print()
#gdpr_df[['gdpr_article', 'priority', 'compliance_gap']].to_string(index=False)
display(gdpr_df[['gdpr_article', 'priority', 'compliance_gap', 'recommended_action']])

GDPR obligations mapped: 8

Priority breakdown:
priority
CRITICAL    4
HIGH        4



,gdpr_article,priority,compliance_gap,recommended_action
0,"Article 5(1)(a) — Lawfulness, Fairness, Transp...",CRITICAL,Missing consent records and absence of a docum...,Implement consent management platform; documen...
1,Article 5(1)(b) — Purpose Limitation,HIGH,No documented purpose specification per proces...,Define and document processing purposes for ea...
2,Article 5(1)(c) — Data Minimisation,HIGH,Overcollection of precision PII beyond model r...,Apply minimisation at collection point: collec...
3,Article 5(1)(e) — Storage Limitation,HIGH,Absence of automated deletion schedule; no doc...,Implement tiered retention: (1) Direct PII — d...
4,Article 6(1) — Lawful Basis for Processing,CRITICAL,No documented LIA for fraud/IP processing; no ...,Document a Legitimate Interests Assessment (LI...
5,Article 17 — Right to Erasure (Right to be For...,CRITICAL,No data subject request (DSR) management syste...,Build DSR workflow with 30-day SLA (Art. 12(3)...
6,Article 22 — Automated Decision-Making,CRITICAL,No evidence of human-in-the-loop for automated...,Implement mandatory human review for all rejec...
7,Article 25 — Data Protection by Design and by ...,HIGH,No Privacy by Design documentation; no field-l...,Adopt layered data architecture: raw (encrypte...
